In [7]:
import cv2
import numpy as np
import os
import yaml
from yaml.loader import SafeLoader

In [8]:
# Load YAML
with open('data.yaml', mode='r') as f:
    data_yaml = yaml.load(f, Loader=SafeLoader)

labels = data_yaml['names']
print(labels)
    




['person', 'car', 'chair', 'bottle', 'pottedplant', 'bird', 'dog', 'sofa', 'horse', 'boat', 'bicycle', 'cat', 'motorbike', 'tvmonitor', 'cow', 'sheep', 'aeroplane', 'train', 'diningtable', 'bus']


In [12]:
# Load YOLO Model
yolo = cv2.dnn.readNetFromONNX('./Model2-20260322T135653Z-1-001/Model2/weights/best.onnx')
yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

In [19]:
# Load the image
img = cv2.imread('./street_image.jpg')
image = img.copy()

#cv2.imshow('image', image)


row, cols, d = image.shape

# get the YOLO prediction from the image
# Step-1 convert image into a square image(array)
max_rc= max(row, cols)
input_image = np.zeros((max_rc, max_rc,3), dtype=np.uint8)
input_image[0:row, 0:cols] = image

# step-2 : get prediction from square array
INPUT_WH_YOLO =640
blob = cv2.dnn.blobFromImage(input_image, 1/255,(INPUT_WH_YOLO, INPUT_WH_YOLO),swapRB=True, crop=False)
yolo.setInput(blob)
preds=yolo.forward()

#cv2.imshow('input_image', input_image)
#cv2.waitKey(0)
#cv2.destroyAllWindows()



In [21]:
print(preds.shape)

(1, 25200, 25)


In [32]:
# Non Maximum Suppression 
# Step-1: filter detection based on confidence (0.4) and propbability score (0.25)
detections = preds[0]
boxes = []
confidences = []
classes = []

# Width and height of the image (input_image)
image_w, image_h = input_image.shape[:2]
x_factor = image_w/INPUT_WH_YOLO
y_factor = image_h/INPUT_WH_YOLO

for i in range(len(detections)):
    row = detections[i]
    confidence = row[4] # Confidence of detection an object
    if confidence > 0.4:
        class_score = row[5:].max() # maximum probability from 20 objects
        class_id =  row[5:].argmax() # get the index postion at whch max probability occurs

        if class_score > 0.25:
            cx, cy, w, h = row[0:4]
            # construct bounding from four values 
            # left, top, width and height
            left = int((cx - 0.5*w)*x_factor)
            top = int((cy - 0.5*h)*y_factor)
            width = int(w*x_factor)
            height = int(h*y_factor)

            box = np.array([left, top, width, height])

            # append values into the list
            confidences.append(confidence)
            boxes.append(box)
            classes.append(class_id)

#clean
boxes_np = np.array(boxes).tolist()
confidences_np = np.array(confidences).tolist()

# NMS
index = cv2.dnn.NMSBoxes(boxes_np, confidences_np, 0.25, 0.45).flatten()
            

In [35]:
index

array([21, 65, 56, 10, 72, 19, 18,  0, 34, 60], dtype=int32)

In [39]:
# Draw the Bounding Box
for ind in index:
    # extract bounding box
    x,y,w,h = boxes_np[ind]
    bb_conf = int(confidences_np[ind]*100)
    classes_id = classes[ind]
    class_name = labels[classes_id]

    text = f'{class_name}: {bb_conf}%'
    print(text)

    cv2.rectangle(image,(x,y),(x+w,y+h),(0,255,0), 2)
    cv2.rectangle(image,(x,y-30),(x+w,y),(255,255,255), -1)

    cv2.putText(image,text,(x,y-10), cv2.FONT_HERSHEY_PLAIN, 0.7, (0,0,0),1)

bird: 84%
person: 84%
person: 65%
person: 57%
bus: 56%
person: 51%
person: 51%
car: 44%
bird: 43%
person: 42%


In [ ]:
cv2.imshow('original', img)
cv2.imshow('yolo_prediction', image)
cv2.waitKey(0)
cv2.destroyAllWindowns()